<a href="https://colab.research.google.com/github/sachin23-an/quant-researcher-practical-lab/blob/main/Quant_Researcher_Practical_Lab_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Quant Researcher — Practical Lab 1
### Foundations → Finding an Alpha → Honest Backtesting

Ye notebook theory wala HTML file (Part 1 Foundations, Part 1B Alpha Research, Part 3 Backtesting) ka **practical companion** hai. Har concept ke baad real code hai — taaki interview mein koi live bole "find me an alpha" ya "backtest this properly", toh haath kaanpe nahi.

**Data:** NSE stocks via `yfinance`. Agar sandbox mein internet na mile (ya rate-limit ho jaaye), notebook automatically synthetic data pe switch kar lega — logic same rehta hai, bas data ka source badalta hai.


In [ ]:
# Setup — run this cell first
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, coint
from scipy.stats import spearmanr
from sklearn.ensemble import RandomForestRegressor

np.random.seed(42)
pd.set_option('display.float_format', lambda x: f'{x:0.4f}')
plt.rcParams['figure.figsize'] = (9, 4)


## 0. Data laao

NSE ke 5 liquid stocks — daily close price, pichhle ~4 saal. Agar `yfinance` fail ho jaaye (no internet / Yahoo rate limit), hum synthetic random-walk prices bana lenge jo realistic correlation structure rakhte hain, taaki aage ka sab code bina error ke chale.


In [ ]:
TICKERS = ['HDFCBANK.NS', 'ICICIBANK.NS', 'RELIANCE.NS', 'TCS.NS', 'INFY.NS']

def load_prices(tickers, start='2021-01-01', end='2025-01-01'):
    try:
        import yfinance as yf
        data = yf.download(tickers, start=start, end=end, auto_adjust=True, progress=False)['Close']
        if data.empty or data.isna().all().all():
            raise ValueError('empty download')
        print('Loaded real NSE data via yfinance.')
        return data.dropna(how='all')
    except Exception as e:
        print(f'yfinance failed ({e}) — falling back to synthetic data.')
        return make_synthetic_prices(tickers)

def make_synthetic_prices(tickers, n_days=1000):
    # correlated random-walk prices, close enough to practice every technique below
    n = len(tickers)
    common_factor = np.cumsum(np.random.randn(n_days)) * 0.5
    dates = pd.bdate_range('2021-01-01', periods=n_days)
    out = {}
    for i, t in enumerate(tickers):
        idio = np.cumsum(np.random.randn(n_days)) * 0.8
        out[t] = 200 + 2*i*10 + common_factor + idio
    return pd.DataFrame(out, index=dates)

prices = load_prices(TICKERS)
prices.tail()


**Kya dekhna hai:** agar `yfinance` chal gaya to real NSE prices honge, warna synthetic — dono case mein neeche ka sab code bina modification ke chalega. Yahi property ek achhi research pipeline mein honi chahiye: data source badle, logic na badle.


## 1. Stationarity — Price vs Return (ADF Test)

Theory file mein (Part 1, Section D) ye padha tha ki price non-stationary hota hai aur return (roughly) stationary. Ab khud test karte hain — Augmented Dickey-Fuller test se.

**Null hypothesis (H0):** series has a unit root (non-stationary).
Low p-value (< 0.05) → reject H0 → series stationary hai.


In [ ]:
def adf_report(series, name):
    series = series.dropna()
    result = adfuller(series, autolag='AIC')
    p_value = result[1]
    verdict = 'STATIONARY' if p_value < 0.05 else 'NON-STATIONARY'
    print(f'{name:12s} ADF p-value = {p_value:.4f}  ->  {verdict}')
    return p_value

stock = prices.columns[0]
price_series = prices[stock]
return_series = prices[stock].pct_change()

adf_report(price_series, 'Price')
adf_report(return_series, 'Return')


**Interpretation:** agar price ka p-value high (non-stationary) aur return ka p-value low (stationary) aaya, to exactly wahi mila jo theory ne predict kiya tha. Interview mein yahi diya jaata hai ek bar: "is this series stationary? prove it" — ab tumhare paas ek function ready hai jo wo prove kar sakta hai.


## 2. Look-Ahead Bias — ek chhoti si bug jo poora backtest jhootha bana deti hai

Theory file ka sabse repeated warning (Part 1, Section H): "#1 thing interviewers probe for". Ab khud ek leaky feature banate hain aur dekhte hain leak kahan hota hai.


In [ ]:
demo = pd.DataFrame({'close': price_series}).dropna()

# BAD: centered rolling window uses 10 days *ahead* of today
demo['leaky_ma20'] = demo['close'].rolling(21, center=True).mean()

# GOOD: strictly backward-looking window
demo['honest_ma20'] = demo['close'].rolling(21).mean()

check_day = 500
future_window_mean = demo['close'].iloc[check_day-10:check_day+11].mean()
print('leaky_ma20 on day', check_day, '=', round(demo['leaky_ma20'].iloc[check_day], 2))
print('equals centered window using FUTURE prices:',
      round(future_window_mean, 2) == round(demo['leaky_ma20'].iloc[check_day], 2))


**Ye observe karo:** `leaky_ma20` exactly wahi number deta hai jo future 10 din ke data se bana — matlab agar ye feature kisi model mein chala gaya, model "future" dekh ke predict kar raha hoga, jo live trading mein possible nahi hai. `honest_ma20` hamesha use karo jab feature engineering kar rahe ho. Ye ek-line ka farak hai (`center=True` vs default) jo poora backtest invalid bana sakta hai.


## 3. Walk-Forward Validation — apne aap ko honestly test karna

Random train/test split time series ke liye illegal hai (future leaks into training). Walk-forward isliye use hota hai — ek reusable splitter function banate hain jo aage har strategy ke liye use hoga.


In [ ]:
def walk_forward_splits(n_rows, train_size, test_size, step=None):
    """Returns list of (train_idx, test_idx) — train hamesha test se pehle aata hai."""
    step = step or test_size
    splits = []
    start = 0
    while start + train_size + test_size <= n_rows:
        train_idx = range(start, start + train_size)
        test_idx = range(start + train_size, start + train_size + test_size)
        splits.append((train_idx, test_idx))
        start += step
    return splits

n = len(prices)
splits = walk_forward_splits(n_rows=n, train_size=252, test_size=63)
print(f'Total rows: {n}, number of walk-forward folds: {len(splits)}')
for i, (tr, te) in enumerate(splits):
    print(f'fold {i}: train rows [{tr.start}:{tr.stop}]  ->  test rows [{te.start}:{te.stop}]')


**Kya dekhna hai:** har fold ka test range hamesha train range ke *baad* aata hai, kabhi overlap nahi karta. Ye wahi function hai jo aage backtest engine (Section 7) ke andar use hoga — ek baar sahi likh liya, baar baar reuse hoga.


## 4. Finding an Alpha — Cross-Sectional Momentum

Ab asli kaam: Part 1B (Alpha Research Loop) ka "hypothesis -> feature -> alpha -> evaluation" cycle khud chalate hain.

**Hypothesis:** jo stocks pichhle 20 din mein zyada chale hain, wo agle 5 din bhi relatively better perform karenge (momentum).


In [ ]:
returns = prices.pct_change()

# Feature: past 20-day return, lagged by 1 day so today's feature never uses today's close
momentum_signal = returns.rolling(20).sum().shift(1)

# Forward return we're trying to predict (5-day horizon)
fwd_return = returns.shift(-5).rolling(5).sum()

momentum_signal.tail(3)


`shift(1)` wahi look-ahead-bias fix hai jo Section 2 mein dekha — signal banate waqt aaj ka return include nahi karna, sirf kal tak ka. Ye habit ab automatic honi chahiye.


## 5. Alpha Evaluation — Information Coefficient (IC) aur Information Ratio (IR)

Theory file ke Part 1B-J mein IC/IR define kiya tha. Ab roz ka IC nikaalte hain: har din, signal ki cross-sectional rank correlation forward return ke saath.


In [ ]:
def compute_ic_series(signal_df, forward_return_df):
    ic_values = []
    dates = []
    for date in signal_df.index:
        s = signal_df.loc[date].dropna()
        f = forward_return_df.loc[date].dropna()
        common = s.index.intersection(f.index)
        if len(common) < 3:
            continue
        ic, _ = spearmanr(s[common], f[common])
        ic_values.append(ic)
        dates.append(date)
    return pd.Series(ic_values, index=dates, name='daily_IC')

ic_series = compute_ic_series(momentum_signal, fwd_return).dropna()

mean_ic = ic_series.mean()
ir = mean_ic / ic_series.std()

print(f'Mean IC : {mean_ic:.4f}')
print(f'IC std  : {ic_series.std():.4f}')
print(f'IR      : {ir:.4f}   (mean IC / std of IC — higher & more stable is better)')

ic_series.rolling(20).mean().plot(title='20-day rolling mean IC — momentum signal')
plt.axhline(0, color='grey', lw=1)
plt.show()


**Reading the output:** IC ka sign aur stability dono important hain — ek bada one-off IC kuch nahi batata, IR (mean/std) batata hai signal kitna *consistently* kaam karta hai. 5 stocks + short history ke saath IC noisy hoga — real research mein 500+ stocks aur saalon ka data use hota hai, yahi number zyada stable milega.


## 6. Mean-Reversion — Cointegration + Pairs Trading (Engle-Granger)

Part 3-E/F ka practical version: do related stocks (yahan HDFC Bank aur ICICI Bank) lo, test karo ki wo cointegrated hain ya nahi, phir spread ka z-score trade karo.


In [ ]:
a, b = prices.columns[0], prices.columns[1]
pair_a, pair_b = prices[a].dropna(), prices[b].dropna()
common_idx = pair_a.index.intersection(pair_b.index)
pair_a, pair_b = pair_a[common_idx], pair_b[common_idx]

score, coint_pvalue, _ = coint(pair_a, pair_b)
print(f'{a} vs {b}  ->  cointegration p-value = {coint_pvalue:.4f}')
print('STATIONARY SPREAD (tradeable pair)' if coint_pvalue < 0.05 else 'NOT cointegrated at 5% — would not trade this pair live')


In [ ]:
# Hedge ratio via OLS, then z-score of the spread
X = sm.add_constant(pair_b)
hedge_ratio = sm.OLS(pair_a, X).fit().params.iloc[1]

spread = pair_a - hedge_ratio * pair_b
z_score = (spread - spread.rolling(60).mean()) / spread.rolling(60).std()

z_score.plot(title=f'Z-score of spread: {a} - {hedge_ratio:.2f} x {b}')
plt.axhline(2, color='r', ls='--', lw=1)
plt.axhline(-2, color='g', ls='--', lw=1)
plt.axhline(0, color='grey', lw=1)
plt.show()


**Trading rule (classic version):** z-score `> 2` → spread historically too wide → short the spread (short A, long B). z-score `< -2` → long the spread. Exit near 0. Hedge ratio OLS se nikala — ye wahi number hai jo decide karta hai kitna B, kitna A ke against lena hai.


## 7. The Backtesting Engine — ek honest measuring instrument

Part 3-B ka core idea: backtest sirf tab sahi hai jab transaction costs, walk-forward split, aur position-at-time-t sab properly handle ho. Ek chhota reusable engine banate hain.


In [ ]:
def backtest_zscore_strategy(z, spread_returns, entry=2.0, exit=0.5, cost_bps=5):
    """z: z-score series. spread_returns: daily return of the spread position.
    cost_bps: round-trip transaction cost in basis points, charged whenever position changes."""
    position = pd.Series(0.0, index=z.index)
    pos = 0.0
    for t in range(1, len(z)):
        if pos == 0 and z.iloc[t] > entry:
            pos = -1.0          # spread too high -> short it
        elif pos == 0 and z.iloc[t] < -entry:
            pos = 1.0           # spread too low -> long it
        elif pos != 0 and abs(z.iloc[t]) < exit:
            pos = 0.0           # mean-reverted -> exit
        position.iloc[t] = pos

    trade_happened = position.diff().abs() > 0
    cost = trade_happened * (cost_bps / 10000)
    strategy_return = position.shift(1) * spread_returns - cost
    return strategy_return.fillna(0), position

spread_returns = spread.pct_change().fillna(0)
strat_ret, position = backtest_zscore_strategy(z_score.fillna(0), spread_returns)

cum_pnl = (1 + strat_ret).cumprod()
cum_pnl.plot(title='Cumulative P&L — pairs z-score strategy (after transaction costs)')
plt.show()

sharpe = strat_ret.mean() / strat_ret.std() * np.sqrt(252)
print(f'Annualized Sharpe (with {5} bps round-trip cost): {sharpe:.2f}')


**Important habit:** `position.shift(1)` — aaj ka return kal ke decided position se aata hai, aaj ke close se liya gaya signal aaj ka return nahi de sakta (wahi look-ahead bias, phir se). Cost sirf tab lagta hai jab position change ho — yehi wo detail hai jo zyada "paper profit" strategies ko real duniya mein negative kar deta hai.


## 8. The Complexity Trap — kya fancier model fayda deta hai?

Part 3-C ka practical test: simple linear momentum signal vs RandomForest — dono ko walk-forward pe test karke dekhte hain ki extra complexity se out-of-sample IC sudharta hai ya sirf overfit karta hai.


In [ ]:
def ic_at_split(signal, forward, idx):
    s = signal.iloc[idx].stack().dropna()
    return s

# Build a simple feature set for one stock (for the ML comparison)
one = prices[prices.columns[0]].pct_change().dropna()
X_feat = pd.DataFrame({
    'mom_5': one.rolling(5).sum(),
    'mom_20': one.rolling(20).sum(),
    'vol_20': one.rolling(20).std(),
}).shift(1).dropna()
y_target = one.shift(-1).reindex(X_feat.index)   # next-day return
data = pd.concat([X_feat, y_target.rename('target')], axis=1).dropna()

folds = walk_forward_splits(len(data), train_size=400, test_size=60)
linear_ics, rf_ics = [], []

for train_idx, test_idx in folds:
    train, test = data.iloc[list(train_idx)], data.iloc[list(test_idx)]
    Xtr, ytr = train[['mom_5','mom_20','vol_20']], train['target']
    Xte, yte = test[['mom_5','mom_20','vol_20']], test['target']

    # simple: just use mom_20 directly as the signal, no fitting
    simple_pred = Xte['mom_20']
    ic_simple, _ = spearmanr(simple_pred, yte)
    linear_ics.append(ic_simple)

    rf = RandomForestRegressor(n_estimators=100, max_depth=3, random_state=0)
    rf.fit(Xtr, ytr)
    rf_pred = rf.predict(Xte)
    ic_rf, _ = spearmanr(rf_pred, yte)
    rf_ics.append(ic_rf)

print(f'Simple momentum  -> mean OOS IC: {np.mean(linear_ics):.4f}')
print(f'RandomForest     -> mean OOS IC: {np.mean(rf_ics):.4f}')


**Yahi hai Complexity Trap ka asli test:** agar RandomForest ka out-of-sample IC simple signal se barabar ya kam hai, to extra complexity sirf training data pe fit ho rahi thi, naya signal nahi de rahi. Interview mein agar koi pooche "why not just use deep learning for everything" — iska jawab yehi number hai, theory nahi.


## Wrap-up

Is notebook mein practically kiya:
1. Stationarity test (ADF) — price vs return
2. Look-ahead bias ko khud create aur detect kiya
3. Reusable walk-forward splitter function
4. Cross-sectional momentum alpha build kiya
5. IC / IR se evaluate kiya
6. Cointegration test + pairs trading z-score strategy
7. Transaction-cost-aware backtest engine
8. Complexity Trap — simple vs ML signal, walk-forward pe compare

**Next notebook (Lab 2)** regime classification (Part 4), options/Greeks/VaR (Part 5), aur risk sizing/Kelly (Part 7/11) practically cover karega — same pattern: theory ke baad turant code.
